In [2]:
import pandas as pd

In [4]:
articles_with_for = pd.read_csv(r"C:\Users\Parsa\Downloads\jcu_articles_with_for_percentages.csv")

In [5]:
articles_with_for

,title,abstract,for_codes,for_codes_with_percentages,primary_for_code,primary_for_percentage,total_percentage_sum,doi,publication,publisher,date,uri,eprintid
0,Mismatch Between Global Importance of Peatland...,Global peatlands store more carbon than all th...,410401,410401:100,410401.0,100.0,100.0,10.1111/conl.13080,Conservation Letters,Wiley-Blackwell,2025,https://researchonline.jcu.edu.au/id/eprint/84705,84705
1,Burden of unintentional drowning in Indonesia:...,Introduction: A high burden of unintentional f...,420604|420603|420207,420604:30|420603:40|420207:30,420603.0,40.0,100.0,10.1136/ip-2024-045274,Injury Prevention,BMJ Group,2025,https://researchonline.jcu.edu.au/id/eprint/83369,83369
2,Unpacking global consciousness: Identifying th...,Global consciousness is a critical construct i...,NaN,NaN,NaN,NaN,NaN,10.1111/ajsp.70031,Asian Journal of Social Psychology,Wiley,2025,https://researchonline.jcu.edu.au/id/eprint/85989,85989
3,The Protective Factors of Suicide in Agricultu...,"Introduction: Each year, over 700,000 people d...",NaN,NaN,NaN,NaN,NaN,10.1080/1059924X.2024.2426500,Journal of Agromedicine,Taylor & Francis,2025,https://researchonline.jcu.edu.au/id/eprint/84335,84335
4,Social Determinants of Suicide and Suicidality...,Issue Addressed: Nearly a million people die b...,520304|420603,520304:30|420603:70,420603.0,70.0,100.0,10.1002/hpja.70030,Health Promotion Journal of Australia,CSIRO Publishing,2025,https://researchonline.jcu.edu.au/id/eprint/86373,86373
...,...,...,...,...,...,...,...,...,...,...,...,...,...
634,How science contributes to environmental repor...,This article examines the role of science in e...,470103,470103:100,470103.0,100.0,100.0,10.1023/A:1020762813548,The Environmentalist,Springer,2002,https://researchonline.jcu.edu.au/id/eprint/79929,79929
635,Global amphibian population declines,The decline and disappearance of relatively un...,69999,69999:100,69999.0,100.0,100.0,10.1038/35087658,Nature,Nature Publishing Group,2001,https://researchonline.jcu.edu.au/id/eprint/13354,13354
636,Modeling the global carbon cycle with a gas hy...,"The ""Latest Paleocene Thermal Maximum"" (or LPT...",40502,40502:100,40502.0,100.0,100.0,10.1029/GM124,Geophysical Monograph Series,American Geophysical Union,2001,https://researchonline.jcu.edu.au/id/eprint/14320,14320
637,Coastal sedimentary research examines critical...,An international conference was held recently ...,40310,40310:100,40310.0,100.0,100.0,10.1029/00EO00125,EOS,American Geophysical Union,2000,https://researchonline.jcu.edu.au/id/eprint/12855,12855


In [ ]:
import pandas as pd
from tqdm import tqdm  # Import tqdm
import concurrent.futures
import time
import ollama

#GOLEN CODE
#Have I ever told you the definition of INSANITY?????????????????????

# Function to process each row
def classify_article(row, index):
    abstract = row['abstract']
    title = row['title']
    system_prompt = (
    "You are an expert in academic article classification.\n"
    "Your task is to classify academic articles into one of the following 23 categories:\n"
    "- Agricultural, Veterinary and Food Sciences\n"
    "- Biological Sciences\n"
    "- Biomedical and Clinical Sciences\n"
    "- Built Environment and Design\n"
    "- Chemical Sciences\n"
    "- Commerce, Management, Tourism and Services\n"
    "- Creative Arts and Writing\n"
    "- Earth Sciences\n"
    "- Economics\n"
    "- Education\n"
    "- Engineering\n"
    "- Environmental Sciences\n"
    "- Health Sciences\n"
    "- History, Heritage and Archaeology\n"
    "- Human Society\n"
    "- Indigenous Studies\n"
    "- Information and Computing Sciences\n"
    "- Language, Communication and Culture\n"
    "- Law and Legal Studies\n"
    "- Mathematical Sciences\n"
    "- Philosophy and Religious Studies\n"
    "- Physical Sciences\n"
    "- Psychology\n"

    "You should provide only the category name."
    )

    user_prompt = f"""
    Title: "{title}"
    Abstract: "{abstract}"

    Based on this information, classify the article into one of the following 23 categories:
    - Agricultural, Veterinary and Food Sciences
    - Biological Sciences
    - Biomedical and Clinical Sciences
    - Built Environment and Design
    - Chemical Sciences
    - Commerce, Management, Tourism and Services
    - Creative Arts and Writing
    - Earth Sciences
    - Economics
    - Education
    - Engineering
    - Environmental Sciences
    - Health Sciences
    - History, Heritage and Archaeology
    - Human Society
    - Indigenous Studies
    - Information and Computing Sciences
    - Language, Communication and Culture
    - Law and Legal Studies
    - Mathematical Sciences
    - Philosophy and Religious Studies
    - Physical Sciences
    - Psychology
    Provide only the category name.
    """
    #print(f"Started processing index {index}")
    start_time = time.time()
    response = ollama.chat(
        model="4o-mini",
        messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
        ]
    )
    try:
        llama_classification = [response["message"]["content"], index]
    except KeyError:
        print(f"Error processing index {index}: KeyError")
        llama_classification = [index] #Add index to make sure the code won't mess up the indexing
    end_time = time.time()
    #print(f"Finished processing index {index}")
    return llama_classification

# Load the Excel file
articles = pd.read_csv("jcu_articles_with_for_percentages.csv")

start_index = 0
counter = 1


classifications = []
start_time = time.time()

# Multithreading for processing with tqdm
with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
    # Wrap the iterator with tqdm
    futures = {
        executor.submit(classify_article, row, index): index
        for index, row in tqdm(articles.iterrows(), total=len(articles), desc=f"Batch {counter} Processing")
    }
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Classification Progress"):
        index = futures[future]
        try:
            classifications.append(future.result())
        except Exception as e:
            print(f"Error processing index {index}: {e}")

total_duration = time.time() - start_time
print(f"Total processing time for {len(articles)} rows: {total_duration:.2f} seconds")

# Process and align classifications
classifications.sort(key=lambda tup: tup[-1])  # Sort to maintain index order
classified_categories = [cls[0] for cls in classifications]  # Extract the category only

# Add classifications as a new column to the DataFrame
articles['4omini_classified'] = classified_categories

# Save to CSV
output_file = f'4omini_FoR_Batch{counter}.csv'
articles.to_csv(output_file, index=False)
print(f"Batch {counter} saved to {output_file}")

In [1]:
FoR_code_dict = {
    "AGRICULTURAL, VETERINARY AND FOOD SCIENCES":30,
    "BIOLOGICAL SCIENCES":31,
    "BIOMEDICAL AND CLINICAL SCIENCES":32,
    "BUILT ENVIRONMENT AND DESIGN":33,
    "CHEMICAL SCIENCES":34,
    "COMMERCE, MANAGEMENT, TOURISM AND SERVICES":35,
    "CREATIVE ARTS AND WRITING":36,
   	"EARTH SCIENCES":37,
    "ECONOMICS":38,
   	"EDUCATION":39,
    "ENGINEERING":40,
    "ENVIRONMENTAL SCIENCES":41,
    "HEALTH SCIENCES":42,
    "HISTORY, HERITAGE AND ARCHAEOLOGY":43,
    "HUMAN SOCIETY":44,
   	"INDIGENOUS STUDIES":45,
    "INFORMATION AND COMPUTING SCIENCES":46,
    "LANGUAGE, COMMUNICATION AND CULTURE":47,
    "LAW AND LEGAL STUDIES":48,
    "MATHEMATICAL SCIENCES":49,
    "PHILOSOPHY AND RELIGIOUS STUDIES":50,
    "PHYSICAL SCIENCES":51,
    "PSYCHOLOGY":52
}

In [3]:
articles_FoR_v2 = pd.read_csv(r"C:\Users\Parsa\Downloads\jcu_articles_with_for_percentages_v2.csv")

In [4]:
articles_FoR_v2

,title,abstract,for_codes,for_codes_with_percentages,primary_for_code,primary_for_percentage,total_percentage_sum,doi,publication,publisher,date,uri,eprintid
0,Mismatch Between Global Importance of Peatland...,Global peatlands store more carbon than all th...,410401,410401:100,410401,100,100.0,10.1111/conl.13080,Conservation Letters,Wiley-Blackwell,2025,https://researchonline.jcu.edu.au/id/eprint/84705,84705
1,Burden of unintentional drowning in Indonesia:...,Introduction: A high burden of unintentional f...,420604|420603|420207,420604:30|420603:40|420207:30,420603,40,100.0,10.1136/ip-2024-045274,Injury Prevention,BMJ Group,2025,https://researchonline.jcu.edu.au/id/eprint/83369,83369
2,Unpacking global consciousness: Identifying th...,Global consciousness is a critical construct i...,NaN,NaN,NaN,NaN,NaN,10.1111/ajsp.70031,Asian Journal of Social Psychology,Wiley,2025,https://researchonline.jcu.edu.au/id/eprint/85989,85989
3,The Protective Factors of Suicide in Agricultu...,"Introduction: Each year, over 700,000 people d...",NaN,NaN,NaN,NaN,NaN,10.1080/1059924X.2024.2426500,Journal of Agromedicine,Taylor & Francis,2025,https://researchonline.jcu.edu.au/id/eprint/84335,84335
4,Social Determinants of Suicide and Suicidality...,Issue Addressed: Nearly a million people die b...,520304|420603,520304:30|420603:70,420603,70,100.0,10.1002/hpja.70030,Health Promotion Journal of Australia,CSIRO Publishing,2025,https://researchonline.jcu.edu.au/id/eprint/86373,86373
...,...,...,...,...,...,...,...,...,...,...,...,...,...
634,How science contributes to environmental repor...,This article examines the role of science in e...,470103,470103:100,470103,100,100.0,10.1023/A:1020762813548,The Environmentalist,Springer,2002,https://researchonline.jcu.edu.au/id/eprint/79929,79929
635,Global amphibian population declines,The decline and disappearance of relatively un...,69999,69999:100,69999,100,100.0,10.1038/35087658,Nature,Nature Publishing Group,2001,https://researchonline.jcu.edu.au/id/eprint/13354,13354
636,Modeling the global carbon cycle with a gas hy...,"The ""Latest Paleocene Thermal Maximum"" (or LPT...",40502,40502:100,40502,100,100.0,10.1029/GM124,Geophysical Monograph Series,American Geophysical Union,2001,https://researchonline.jcu.edu.au/id/eprint/14320,14320
637,Coastal sedimentary research examines critical...,An international conference was held recently ...,40310,40310:100,40310,100,100.0,10.1029/00EO00125,EOS,American Geophysical Union,2000,https://researchonline.jcu.edu.au/id/eprint/12855,12855


In [ ]:
import pandas as pd
import numpy as np
import re

# === 1) INPUTS ===
CSV_PATH = "jcu_articles_with_for_percentages_v2.csv"   # <-- change to your file
PRED_COL = "4omini_classified"                          # model output column
PRIMARY_COL = "primary_for_code"                        # ground-truth (may be "AB|CD" or a single code)

# === 2) FoR division mapping (given) ===
FoR_code_dict = {
    "AGRICULTURAL, VETERINARY AND FOOD SCIENCES":30,
    "BIOLOGICAL SCIENCES":31,
    "BIOMEDICAL AND CLINICAL SCIENCES":32,
    "BUILT ENVIRONMENT AND DESIGN":33,
    "CHEMICAL SCIENCES":34,
    "COMMERCE, MANAGEMENT, TOURISM AND SERVICES":35,
    "CREATIVE ARTS AND WRITING":36,
    "EARTH SCIENCES":37,
    "ECONOMICS":38,
    "EDUCATION":39,
    "ENGINEERING":40,
    "ENVIRONMENTAL SCIENCES":41,
    "HEALTH SCIENCES":42,
    "HISTORY, HERITAGE AND ARCHAEOLOGY":43,
    "HUMAN SOCIETY":44,
    "INDIGENOUS STUDIES":45,
    "INFORMATION AND COMPUTING SCIENCES":46,
    "LANGUAGE, COMMUNICATION AND CULTURE":47,
    "LAW AND LEGAL STUDIES":48,
    "MATHEMATICAL SCIENCES":49,
    "PHILOSOPHY AND RELIGIOUS STUDIES":50,
    "PHYSICAL SCIENCES":51,
    "PSYCHOLOGY":52
}
LABEL_TO_DIV = {k.upper(): f"{v:02d}" for k, v in FoR_code_dict.items()}

# === 3) Helpers ===
def label_to_div2(label: str):
    """Map model label (string) to a 2-digit division code, or None if unmapped."""
    if not isinstance(label, str) or not label.strip():
        return None
    return LABEL_TO_DIV.get(label.strip().upper())

def code_to_div2(code: str):
    """
    Extract the first two digits from a FoR code string like '410401' -> '41'.
    Returns None if not found.
    """
    if not isinstance(code, str):
        code = "" if pd.isna(code) else str(code)
    m = re.search(r"\d{2}", code)
    return m.group(0) if m else None

def primary_prefixes(value):
    """
    Parse PRIMARY_COL which may be a single code '410401' or a tie '310101|310102'.
    Return a set of 2-digit prefixes (e.g., {'41'} or {'31'}).
    """
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    parts = [p.strip() for p in value.split("|") if p.strip()]
    prefixes = {code_to_div2(p) for p in parts if code_to_div2(p) is not None}
    return prefixes if prefixes else None

# === 4) Load & compute ===
df = pd.read_csv(CSV_PATH, dtype=str)  # read everything as string; safer for codes
df["pred_div2"] = df[PRED_COL].apply(label_to_div2)
df["primary_div2_set"] = df[PRIMARY_COL].apply(primary_prefixes)

# evaluable rows = have both a mapped prediction and at least one primary prefix
evaluable = df[df["pred_div2"].notna() & df["primary_div2_set"].notna()].copy()

def is_correct(row):
    return row["pred_div2"] in row["primary_div2_set"]

evaluable["correct"] = evaluable.apply(is_correct, axis=1)

# === 5) Metrics ===
n_total = len(df)
n_evaluable = len(evaluable)
n_correct = int(evaluable["correct"].sum())
accuracy = (n_correct / n_evaluable) if n_evaluable else float("nan")

print(f"Total rows                : {n_total}")
print(f"Evaluable rows (have GT & prediction): {n_evaluable}")
print(f"Correct predictions       : {n_correct}")
print(f"Accuracy (div2 match)     : {accuracy:.4f}")

# Optional: per-division accuracy (by ground-truth division)
# We expand primary_div2_set to multiple rows if needed (for 50/50 ties)
rows = []
for _, r in evaluable.iterrows():
    for gt_div in r["primary_div2_set"]:
        rows.append({"gt_div2": gt_div, "pred_div2": r["pred_div2"], "correct": r["pred_div2"] == gt_div})
per_div = pd.DataFrame(rows)
if not per_div.empty:
    per_acc = per_div.groupby("gt_div2")["correct"].mean().sort_index()
    print("\nPer-division accuracy (by ground-truth division):")
    print(per_acc.to_string())

# Optional: save back the correctness flags
# df = df.merge(evaluable[["correct"]], left_index=True, right_index=True, how="left")
# df.to_csv("with_accuracy_flags.csv", index=False)
